In [177]:
# fazendo importações
import os
from random import choice, choices, randint, triangular, sample
from dotenv import load_dotenv
from faker import Faker
import pandas as pd
from sqlalchemy import create_engine
from re import sub
from unicodedata import normalize
from string import ascii_lowercase
from argon2 import PasswordHasher

In [178]:
# conectando com o banco de dados
load_dotenv()

engine = create_engine(os.getenv("DATABASE_URL"))
faker = Faker("pt_BR")

In [179]:
# constantes
QTD_COLABORADORES = 50
QTD_EMPRESAS = 5

In [180]:
# utilitários
quantidade_dependencias = [1, 2, 3, 4]

provedores_email = ['gmail.com', 'hotmail.com', 'outlook.com', 'yahoo.com.br']

tamanho_empresa = ['PEQUENA', 'MEDIA', 'GRANDE']

setores = ["Tecnologia", "Saúde", "Educação", "Financeiro", "Indústria", "Varejo", "Atacado", "Logística", "Transporte", "Construção Civil", "Agronegócio", "Alimentação", "Bebidas", "Farmacêutico", "Químico", "Metalurgia", "Mineração", "Energia", "Telecomunicações", "Turismo", "Hotelaria", "Restaurante", "Imobiliário", "Marketing", "Publicidade", "Recursos Humanos", "Consultoria", "Jurídico", "Contabilidade", "Seguros", "Automotivo", "Têxtil", "Moda", "Cosméticos", "Beleza", "Entretenimento", "Mídia", "E-commerce", "Serviços", "Segurança"]

areas = ["Administrativo","Financeiro","Contabilidade","Recursos Humanos","Departamento Pessoal","Comercial","Vendas","Marketing","Atendimento","SAC","Tecnologia da Informação","Desenvolvimento","Infraestrutura","Segurança da Informação","Jurídico","Compras","Logística","Almoxarifado","Produção","Qualidade","Engenharia","Projetos","Operações","Manutenção","Auditoria","Controladoria","Planejamento","B.I.","Comunicação","Diretoria","Presidência"]

ddds = [11,12,13,14,15,16,17,18,19,21,22,24,27,28,31,32,33,34,35,37,38,41,42,43,44,45,46,47,48,49,51,53,54,55,61,62,63,64,65,66,67,68,69,71,73,74,75,77,79,81,82,83,84,85,86,87,88,89,91,92,93,94,95,96,97,98,99]

ph = PasswordHasher(time_cost=4, memory_cost=262144, parallelism=2)
PEPPER = os.getenv('PEPPER')

In [181]:
# funções
def normalizar(valor: str) -> str:
    return sub(r'[^a-zA-Z0-9]+', '.', normalize('NFKD', valor).encode('ASCII', 'ignore').decode('ASCII').lower())

def apenas_numeros(valor: str) -> str:
    return sub(r'\D', '', valor)

def gerar_lista_ids(limite: int) -> list:
    return list(range(1, limite + 1))

def gerar_valor_aleatorio(qtd_caracteres: int) -> str:
    return ''.join(choices(ascii_lowercase, k=qtd_caracteres))

def gerar_email(nome_email: str) -> str:
    return f'{normalizar(nome_email)}_{gerar_valor_aleatorio(4)}@{choice(provedores_email)}'

def gerar_telefone() -> str:
    return f'{choice(ddds)}9{randint(1000, 9999)}{randint(1000,9999)}'

def gerar_hash_senha() -> str:
    senha = faker.password(special_chars=True, digits=True, upper_case=True, lower_case=True)
    return ph.hash(senha + PEPPER)


In [182]:
# listas de id
lista_id_empresa = gerar_lista_ids(QTD_EMPRESAS)
lista_id_usuario = gerar_lista_ids(QTD_COLABORADORES)

In [183]:
 # informações de empresa
dados_email_empresa = []
dados_telefone_empresa = []
dados_endereco_empresa = []
dados_empresa = []

for i in range(QTD_EMPRESAS):
    QTD_EMAILS_EMPRESA = choices(quantidade_dependencias, weights=(60, 25, 10, 5), k=1)[0]
    QTD_TELEFONES_EMPRESA = choices(quantidade_dependencias, weights=(60, 25, 10, 5), k=1)[0]
    QTD_ENDERECOS_EMPRESA = choices(quantidade_dependencias, weights=(60, 25, 10, 5), k=1)[0]

    id_empresa = i + 1
    nome = faker.company()

    for j in range(QTD_EMAILS_EMPRESA):
        dados_email_empresa.append({
            'id_empresa': id_empresa,
            'email': gerar_email(nome),
            'principal': True if j == 0 else False
        })

    for j in range(QTD_TELEFONES_EMPRESA):
        dados_telefone_empresa.append({
            'id_empresa': id_empresa,
            'numero_telefone': gerar_telefone(),
            'principal': True if j == 0 else False
        })
    
    for j in range(QTD_ENDERECOS_EMPRESA):
        dados_endereco_empresa.append({
            'id_empresa': id_empresa,
            'cep': apenas_numeros(faker.postcode()),
            'uf': faker.state_abbr(),
            'cidade': faker.city(),
            'bairro': faker.neighborhood(),
            'logradouro': faker.street_name(),
            'numero_endereco': faker.building_number(),
            'principal': True if j == 0 else False
        })

    dados_empresa.append({
        'cnpj': apenas_numeros(faker.cnpj()),
        'nome': nome,
        'tamanho_empresa': choice(tamanho_empresa),
        'setor_empresa': choice(setores),
        'status': 'ATIVO'
    })

In [184]:
# informações de colaborador

def gerar_colaborador(id_empresa: int, tipo: str, id: int):
    qtd_emails = choices(quantidade_dependencias, weights=(80, 20, 0, 0), k=1)[0]
    qtd_telefones = choices(quantidade_dependencias, weights=(80, 20, 0, 0), k=1)[0]

    nome = faker.name()

    for i in range(qtd_emails):
        dados_email_colaborador.append({
            'id_colaborador': id,
            'email': gerar_email(nome),
            'principal': True if i == 0 else False
        })

    for i in range(qtd_telefones):
        dados_telefone_colaborador.append({
            'id_colaborador': id,
            'numero_telefone': apenas_numeros(faker.cellphone_number()),
            'principal': True if i== 0 else False
        })

    dados_usuario.append({
        'id_empresa': id_empresa,
        'nome': nome,
        'email_login': gerar_email(nome),
        'senha_hash': gerar_hash_senha(),
        'tipo_usuario': tipo,
        'status': 'ATIVO'
    })

    dados_colaborador.append({
        'nome': nome,
        'id_empresa': id_empresa,
        'id_usuario': id,
        'cpf': apenas_numeros(faker.cpf()),
        'cargo': faker.job(),
        'area': choice(areas),
        'data_nascimento': faker.date_of_birth(minimum_age=18, maximum_age=70),
        'data_contratacao': faker.date_this_decade(),
        'permissao_gestor': True if tipo == 'GESTOR' or tipo == 'ADMIN' else False,
        'status': 'ATIVO'
    })

dados_email_colaborador = []
dados_telefone_colaborador = []
dados_usuario = []
dados_colaborador = []

id = 1
for id_empresa in lista_id_empresa:
    qtd_colaboradores = randint(4, 29)

    # gerar gestor
    gerar_colaborador(id_empresa, 'GESTOR', id)
    id += 1

    for _ in range(qtd_colaboradores):
        tipo = choices(['COLABORADOR', 'ADMIN'], weights=(95, 5), k=1)[0]
        gerar_colaborador(id_empresa, tipo, id)
        id+=1

In [185]:
# informações do ciclo
dados_ciclo = []
gestor_ciclo = pd.DataFrame(dados_colaborador).groupby('id_empresa').first().reset_index()

for _, gestor in gestor_ciclo.iterrows():
    dados_ciclo.append({
        'id_empresa': gestor['id_empresa'],
        'id_responsavel': gestor['id_usuario'],
        'id_ishikawa_mongo': randint(1, 9999),
        'titulo': faker.sentence(nb_words=5),
        'descricao': faker.paragraph(nb_sentences=10),
        'status': 'CONCLUIDO',
        'data_inicio': faker.past_date(start_date='-1y'),
        'data_estimada_fim': faker.future_date(),
        'data_fim_real': faker.future_date()
    })

In [186]:
# informações de plano de ação
dados_plano_acao = []
id_plano_acao = 1

for id_ciclo, ciclo in enumerate(dados_ciclo, start=1):
    dados_plano_acao.append({
        'id': id_plano_acao,
        'id_ciclo': id_ciclo,
        'nome': faker.sentence(nb_words=5),
        'objetivo': faker.sentence(nb_words=10),
        'prioridade': choice(['BAIXA', 'MEDIA', 'ALTA', 'CRITICA']),
        'status': 'APROVADO',
        'origem': 'MANUAL',
        'criado_por': ciclo['id_responsavel']
    })

    id_plano_acao += 1

In [187]:
# informações do problema
dados_problema = []
id_problema = 1

for id_ciclo, ciclo in enumerate(dados_ciclo, start=1):
    qtd_problema = randint(1, 5)

    for i in range(qtd_problema):
        dados_problema.append({
            'id': id_problema,
            'id_ciclo': id_ciclo,
            'criado_por': ciclo['id_responsavel'],
            'titulo': faker.sentence(nb_words=5),
            'descricao': faker.paragraph(nb_sentences=10),
            'peso': round((1 / qtd_problema), 2),
            'status': 'ABERTO',
            'origem': 'MANUAL',
            'persistente': False,

            # apenas para referência da causa raiz
            'principal': True if i == 0 else False
        })

        id_problema += 1

In [188]:
# informações da causa raiz
dados_causa_raiz = []
planos = {plano['id_ciclo']: plano['id'] for plano in dados_plano_acao}

for problema in dados_problema:
    dados_causa_raiz.append({
        'id_ciclo': problema['id_ciclo'],
        'id_problema': problema['id'],
        'id_plano_acao': planos[problema['id_ciclo']],
        'id_5_porques_mongo': randint(1, 9999),
        'validada_por': problema['criado_por'],
        'descricao': faker.paragraph(nb_sentences=10),
        'origem': 'MANUAL',
        'principal': problema['principal'],
        'aceita': False
    })

In [189]:
# informações de meta
dados_meta = []

for id_ciclo, ciclo in enumerate(dados_ciclo, start=1):
    for _ in range(randint(1, 3)):
        dados_meta.append({
            'id_ciclo': id_ciclo,
            'objetivo': faker.sentence(nb_words=10),
            'valor_base': randint(1000, 100000),
            'valor_alvo': randint(1000, 100000),
            'unidade': choice(['R$', 'US$', 'EUR']),
            'prazo': faker.future_date(),
            'status': choice(['PARCIALMENTE_ATINGIDA', 'ATINGIDA', 'NAO_ATINGIDA']),
            'prioridade': choice(['BAIXA', 'MEDIA', 'ALTA', 'CRITICA']),
            'area': choice(areas),
            'categoria': faker.word()
        })

In [190]:
# informações de 5W2H
dados_5w2h = []

for id_plano_acao, plano_acao in enumerate(dados_plano_acao, start=1):
    dados_5w2h.append({
        'id_plano_acao': id_plano_acao,
        'id_who_responsavel': plano_acao['criado_por'],
        'what_acao': faker.sentence(nb_words=10),
        'why_justificativa': faker.sentence(nb_words=10),
        'where_local': faker.sentence(nb_words=10),
        'when_fim': faker.future_date(),
        'how_modo_execucao': faker.sentence(nb_words=10),
        'how_much_custo': randint(1000, 100000),
    })

In [191]:
# informações de tarefa
dados_tarefa = []

for id_plano_acao, plano_acao in enumerate(dados_plano_acao, start=1):
    for _ in range(randint(1, 10)):
        dados_tarefa.append({
            'id_plano_acao': id_plano_acao,
            'id_responsavel': plano_acao['criado_por'],
            'titulo': faker.sentence(nb_words=5),
            'descricao': faker.sentence(nb_words=10),
            'prioridade': choice(['BAIXA', 'MEDIA', 'ALTA', 'CRITICA']),
            'status': 'PENDENTE',
            'data_fim_prevista': faker.future_date()
        })

In [192]:
# informações de treinamento
dados_treinamento = []

for id_ciclo, ciclo in enumerate(dados_ciclo, start=1):
    dados_treinamento.append({
        'id_ciclo': id_ciclo,
        'id_anexo_mongo': randint(1, 9999),
        'id_responsavel': ciclo['id_responsavel'],
        'titulo': faker.sentence(nb_words=5),
        'descricao': faker.sentence(nb_words=10),
        'data_treinamento': faker.future_date(),
        'obrigatorio': choice([True, False])
    })

In [193]:
# dados de verificação do resultado
dados_verificacao = []

for id_ciclo, ciclo in enumerate(dados_ciclo, start=1):
    dados_verificacao.append({
        'id_ciclo': id_ciclo,
        'criado_por': ciclo['id_responsavel'],
        'status': choice(['APROVADO', 'REPROVADO', 'NAO_VERIFICADO', 'PARCIAL']),
        'resumo': faker.sentence(nb_words=10),
        'observacao': faker.sentence(nb_words=10)
    })

In [194]:
# dados de efeito secundário
dados_efeito_secundario = []

for id_verificacao, verificacao in enumerate(dados_verificacao, start=1):
    for _ in range(randint(1, 10)):
        dados_efeito_secundario.append({
            'id_verificacao_resultado': id_verificacao,
            'descricao': faker.sentence(nb_words=10),
            'peso': round(triangular(0.01, 0.99, 0.5), 2),
            'impacto_estimado': faker.sentence(nb_words=10),
            'tipo': choice(['POSITIVO', 'NEGATIVO'])
        })

In [195]:
# criando dataframes
df_email_empresa = pd.DataFrame(dados_email_empresa)
df_telefone_empresa = pd.DataFrame(dados_telefone_empresa)
df_endereco_empresa = pd.DataFrame(dados_endereco_empresa)
df_empresa = pd.DataFrame(dados_empresa)
df_email_colaborador = pd.DataFrame(dados_email_colaborador)
df_telefone_colaborador = pd.DataFrame(dados_telefone_colaborador)
df_usuario = pd.DataFrame(dados_usuario)
df_colaborador = pd.DataFrame(dados_colaborador)
df_ciclo = pd.DataFrame(dados_ciclo)
df_plano_acao = pd.DataFrame(dados_plano_acao)
df_causa_raiz = pd.DataFrame(dados_causa_raiz)
df_efeito_secundario = pd.DataFrame(dados_efeito_secundario)
df_meta = pd.DataFrame(dados_meta)
df_5w2h = pd.DataFrame(dados_5w2h)
df_problema = pd.DataFrame(dados_problema)
df_tarefa = pd.DataFrame(dados_tarefa)
df_treinamento = pd.DataFrame(dados_treinamento)
df_verificacao = pd.DataFrame(dados_verificacao)

In [196]:
# dados das tabelas de ligação
usuarios_empresa = df_colaborador.groupby('id_empresa')['id_usuario'].apply(list).to_dict()
ciclo_empresa = {id_ciclo: ciclo['id_empresa'] for id_ciclo, ciclo in enumerate(dados_ciclo, start=1)}

# usuários por ciclo
dados_usuario_ciclo = []
dados_meta_responsavel = []
dados_usuario_treinamento = []
dados_priorizacao = []

for id_ciclo, ciclo in enumerate(dados_ciclo, start=1):
    colaborador_empresa = usuarios_empresa[ciclo['id_empresa']]
    gestor = ciclo['id_responsavel']

    dados_usuario_ciclo.append({
        'id_usuario': gestor,
        'id_ciclo': id_ciclo,
        'papel_ciclo': 'RESPONSAVEL'
    })

    outros = [c for c in colaborador_empresa if c != gestor]
    for id_usuario in outros:
        dados_usuario_ciclo.append({
            'id_usuario': id_usuario,
            'id_ciclo': id_ciclo,
            'papel_ciclo': choices(['PARTICIPANTE', 'EXECUTOR', 'VALIDADOR', 'OBSERVADOR'], weights=[70, 10, 10, 10], k=1)[0]
        })

for id_meta, meta in enumerate(dados_meta, start=1):
    colaborador_empresa = usuarios_empresa[ciclo_empresa[meta['id_ciclo']]]

    for id_usuario in colaborador_empresa:
        dados_meta_responsavel.append({
            'id_meta': id_meta,
            'id_usuario': id_usuario,
        })

for id_treinamento, treinamento in enumerate(dados_treinamento, start = 1):
    colaborador_empresa = usuarios_empresa[ciclo_empresa[treinamento['id_ciclo']]]

    for id_usuario in colaborador_empresa:
        dados_usuario_treinamento.append({
            'id_treinamento': id_treinamento,
            'id_usuario': id_usuario,
            'obrigatorio': treinamento['obrigatorio'],
            'status': choice(['PENDENTE', 'CONFIRMADO', 'CONCLUIDO']),
            'terminado_em': faker.future_date()
        })

for problema in dados_problema:
    colaborador_empresa = usuarios_empresa[ciclo_empresa[problema['id_ciclo']]]

    for _, id_usuario in enumerate(colaborador_empresa, start=1):
        posicao = randint(1, 10)

        dados_priorizacao.append({
            'id_problema': problema['id'],
            'id_usuario': id_usuario,
            'posicao': posicao,
            'peso_calculado': round(1 / posicao, 2)
        })

# criando dataframes
df_meta_responsavel = pd.DataFrame(dados_meta_responsavel)
df_priorizacao_problema = pd.DataFrame(dados_priorizacao)
df_usuario_treinamento = pd.DataFrame(dados_usuario_treinamento)
df_usuario_ciclo = pd.DataFrame(dados_usuario_ciclo)

In [197]:
tabelas = {
    ('public', 'empresa'): df_empresa,
    ('public', 'usuario_sistema'): df_usuario,
    ('public', 'colaborador'): df_colaborador,
    ('public', 'email_empresa'): df_email_empresa,
    ('public', 'telefone_empresa'): df_telefone_empresa,
    ('public', 'endereco_empresa'): df_endereco_empresa,
    ('public', 'email_colaborador'): df_email_colaborador,
    ('public', 'telefone_colaborador'): df_telefone_colaborador,
    ('pdca', 'ciclo'): df_ciclo,
    ('pdca', 'meta'): df_meta,
    ('pdca', 'plano_acao'): df_plano_acao,
    ('pdca', 'treinamento'): df_treinamento,
    ('pdca', 'verificacao_resultado'): df_verificacao,
    ('pdca', 'problema'): df_problema,
    ('pdca', 'causa_raiz'): df_causa_raiz,
    ('pdca', 'plano_5w2h'): df_5w2h,
    ('pdca', 'efeito_secundario'): df_efeito_secundario,
    ('pdca', 'tarefa'): df_tarefa,
    ('pdca', 'meta_responsavel'): df_meta_responsavel,
    ('pdca', 'priorizacao_problema_usuario'): df_priorizacao_problema,
    ('pdca', 'usuario_ciclo'): df_usuario_ciclo,
    ('pdca', 'usuario_treinamento'): df_usuario_treinamento
}

In [198]:
# tratamento dos dados
df_problema.drop(columns=['principal'], inplace=True)

In [199]:
print('Iniciando conexão com o banco de dados')
for (schema, tabela), df in tabelas.items():
    try:
        df.to_sql(name = tabela, schema = schema, con = engine, if_exists = 'append', index = False)
    except Exception as e:
        print(f'Um erro no banco de dados ocorreu: {e}')
        break

print('Conexão com o banco de dados finalizada com sucesso')

Iniciando conexão com o banco de dados
Conexão com o banco de dados finalizada com sucesso
